In [7]:
# Re-import necessary libraries due to environment reset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datasketch import MinHash, MinHashLSH

# Reload data
student_df = pd.read_csv("StudentAddresses-2016-2024.csv")
rental_df = pd.read_csv("current_rental_registrations.csv")

C:\Users\bens0\AppData\Local\Temp\ipykernel_18152\2016703708.py:8: DtypeWarning: Columns (4,7,8) have mixed types. Specify dtype option on import or set low_memory=False.
  student_df = pd.read_csv("StudentAddresses-2016-2024.csv")


In [8]:
# Fix ZIP codes
student_df['6e. zip'] = student_df['6e. zip'].astype(str).apply(lambda z: (z if z.startswith('0') else '0' + z)[:5])

# Define suffix abbreviation dictionary
suffix_abbrev =  {
    "STREET": "ST", "ST": "ST", "ST.": "ST", "STR": "ST", "STEET": "ST", "STREET,": "ST", "STREET.": "ST",
    "AVENUE": "AVE", "AVE": "AVE", "AVE.": "AVE", "AV": "AVE",
    "BOULEVARD": "BLVD", "BLVD": "BLVD", "BLVD.": "BLVD",
    "ROAD": "RD", "RD": "RD", "RD.": "RD", "ROA": "RD",
    "DRIVE": "DR", "DR": "DR", "DR.": "DR", "DRIV": "DR",
    "COURT": "CT", "CT": "CT", "CT.": "CT", "COURT," : "CT",
    "LANE": "LN", "LN": "LN", "LN.": "LN",
    "PLACE": "PL", "PL": "PL", "PL.": "PL", "PLAZA": "PL",
    "TERRACE": "TER", "TER": "TER", "TER.": "TER",
    "CIRCLE": "CIR", "CIR": "CIR", "CIRCUIT": "CIR",
    "PARKWAY": "PKWY", "PKWY": "PKWY", "PK": "PKWY",
    "SQUARE": "SQ", "SQ": "SQ", "SQ.": "SQ",
    "WAY": "WAY", "WAY.": "WAY", "WY": "WAY",
    "HIGHWAY": "HWY", "HWY": "HWY",
    "ALLEY": "ALY", "ALY": "ALY"
}

def fix_shifted_street_columns(row):
    # CASE 1: Street name is missing, suffix contains both
    if pd.isna(row['6b. street name']) and pd.notna(row['6c. street suffix']):
        parts = str(row['6c. street suffix']).strip().upper().split()
        if len(parts) >= 2:
            row['6b. street name'] = " ".join(parts[:-1])
            row['6c. street suffix'] = parts[-1]

    # CASE 2: Suffix is missing, street name contains both
    elif pd.notna(row['6b. street name']) and pd.isna(row['6c. street suffix']):
        parts = str(row['6b. street name']).strip().upper().split()
        if len(parts) >= 2:
            row['6b. street name'] = " ".join(parts[:-1])
            row['6c. street suffix'] = parts[-1]

    return row

student_df = student_df.apply(fix_shifted_street_columns, axis=1)

# Student address standardization
def standardize_student_address_no_unit(row):
    street_number = str(row['6a. street #']).strip()

    # Assume cleaned inputs (no NaNs or merged fields)
    street_name = str(row['6b. street name']).strip().upper()
    suffix = str(row['6c. street suffix']).strip().upper()
    suffix = suffix_abbrev.get(suffix, suffix)  # Normalize suffix
    zip_code = str(row['6e. zip']).strip()

    # Build the address string
    address_parts = [street_number, street_name, suffix]
    address = " ".join([part for part in address_parts if part]) + f", MA {zip_code}"

    return address.upper().replace("  ", " ").strip()  

student_df["standardized_address"] = student_df.apply(standardize_student_address_no_unit, axis=1)

# Rental address cleaning
def clean_registered_address(addr):
    if pd.isna(addr):
        return np.nan
    addr = addr.upper().strip().replace("  ", " ")
    zip_code = addr[-5:] if addr[-5:].isdigit() else ""
    addr_main = addr.split(",")[0]
    tokens = addr_main.split()
    if tokens and len(tokens[-1]) <= 5 and any(c.isdigit() for c in tokens[-1]) and not tokens[-1][-1].isalpha():
        tokens = tokens[:-1]
    if len(tokens) >= 2 and tokens[-1] in suffix_abbrev:
        tokens[-1] = suffix_abbrev[tokens[-1]]
    base = " ".join(tokens)
    return f"{base}, MA {zip_code}"

rental_df["standardized_address"] = rental_df["RegisteredAddress"].apply(clean_registered_address)


In [9]:
import re
from datasketch import MinHash, MinHashLSH

# Step 1: Prepare rental address list for LSH
rental_addresses = rental_df['standardized_address'].dropna().unique().tolist()

# Step 2: Build MinHashLSH index from rental addresses
rental_lsh = MinHashLSH(threshold=0.85, num_perm=128)
rental_minhashes = {}

for addr in rental_addresses:
    mh = MinHash(num_perm=128)
    for token in addr.lower().split():
        mh.update(token.encode('utf8'))
    rental_minhashes[addr] = mh
    rental_lsh.insert(addr, mh)

# Step 3: Extract street number from address
def extract_street_number(address):
    """Extracts the leading number from the address (ignores ranges like 22-24)."""
    match = re.match(r'^(\d+)', address)
    return int(match.group(1)) if match else None

# Step 4: Deduplicate student addresses
student_df_unique = student_df.drop_duplicates(subset=["standardized_address"]).copy()

# Step 5: Query LSH with number check
def query_lsh_with_number_check(student_address):
    mh = MinHash(num_perm=128)
    for token in student_address.lower().split():
        mh.update(token.encode('utf8'))

    results = rental_lsh.query(mh)
    student_num = extract_street_number(student_address)

    for match in results:
        rental_num = extract_street_number(match)
        if rental_num == student_num:
            return match

    return None

# Step 6: Apply match logic to student addresses
student_df_unique['matched_address'] = student_df_unique['standardized_address'].apply(query_lsh_with_number_check)
student_df_unique['registration_status'] = student_df_unique['matched_address'].apply(
    lambda x: 'Registered (Fuzzy)' if pd.notna(x) else 'Unregistered'
)


In [10]:
student_df_unique.head(50)

,6a. street #,6b. street name,6c. street suffix,6d. unit #,6e. zip,7. undergraduate (u) or graduate (g),8. full-time (ft) or part-time (pt),9. at-home or not-at-home,9. 5 or more undergrads/unit (y/n),university,year,standardized_address,matched_address,registration_status
0,10,Higgins,ST,NaN,02134,U,FT,NaN,NaN,Emmanuel College,2018-2019,"10 HIGGINS ST, MA 02134",None,Unregistered
2,1189,Commonwealth,AVE,6,02134,U,FT,NaN,NaN,Emmanuel College,2018-2019,"1189 COMMONWEALTH AVE, MA 02134","1189 COMMONWEALTH AVE, MA 02134",Registered (Fuzzy)
3,12,Glenville,AVE,NaN,02134,U,FT,NaN,NaN,Emmanuel College,2018-2019,"12 GLENVILLE AVE, MA 02134",None,Unregistered
5,12,Saunders,ST,NaN,02134,U,FT,NaN,NaN,Emmanuel College,2018-2019,"12 SAUNDERS ST, MA 02134",None,Unregistered
6,1251,Commonwealth,AVE,3,02134,U,FT,NaN,NaN,Emmanuel College,2018-2019,"1251 COMMONWEALTH AVE, MA 02134","1251 COMMONWEALTH AVE, MA 02134",Registered (Fuzzy)
7,17,Highgate,ST,NaN,02134,U,FT,NaN,NaN,Emmanuel College,2018-2019,"17 HIGHGATE ST, MA 02134","17 HIGHGATE ST, MA 02134",Registered (Fuzzy)
8,28,Linden,ST,NaN,02134,U,FT,NaN,NaN,Emmanuel College,2018-2019,"28 LINDEN ST, MA 02134",None,Unregistered
9,28,Quint,AVE,48,02134,U,FT,NaN,NaN,Emmanuel College,2018-2019,"28 QUINT AVE, MA 02134","28 QUINT AVE, MA 02134",Registered (Fuzzy)
10,40,Brainerd,RD,NaN,02134,U,FT,NaN,NaN,Emmanuel College,2018-2019,"40 BRAINERD RD, MA 02134",None,Unregistered
11,49,Pratt,ST,NaN,02134,U,FT,NaN,NaN,Emmanuel College,2018-2019,"49 PRATT ST, MA 02134",None,Unregistered


In [11]:
student_df_unique.loc[164800:165000]

,6a. street #,6b. street name,6c. street suffix,6d. unit #,6e. zip,7. undergraduate (u) or graduate (g),8. full-time (ft) or part-time (pt),9. at-home or not-at-home,9. 5 or more undergrads/unit (y/n),university,year,standardized_address,matched_address,registration_status
164824,2021,COMMOWEALTH,AVENUE,NaN,02135,U,FT,Not-at-Home,NaN,Boston College,2023-2024,"2021 COMMOWEALTH AVE, MA 02135",None,Unregistered
164841,31-35,SOUTH,STREET,9,02135,U,FT,Not-at-Home,NaN,Boston College,2023-2024,"31-35 SOUTH ST, MA 02135",None,Unregistered
164848,1,PORTINA,RD.,#1,02135,G,Full Time,Not-at-Home,NaN,Boston College,2023-2024,"1 PORTINA RD, MA 02135",None,Unregistered
164850,1,WASHINGTON,MALL,1443,02108,G,Part Time,Not-at-Home,NaN,Boston College,2023-2024,"1 WASHINGTON MALL, MA 02108",None,Unregistered
164854,10,GARDEN COURT,STREET,NaN,02113,G,Part Time,Not-at-Home,NaN,Boston College,2023-2024,"10 GARDEN COURT ST, MA 02113","10 GARDEN COURT ST, MA 02113",Registered (Fuzzy)
164862,100,MAIN,ST,Apt 5,02129,G,Part Time,Not-at-Home,NaN,Boston College,2023-2024,"100 MAIN ST, MA 02129","100 MAIN ST, MA 02129",Registered (Fuzzy)
164865,101,CHESTNUT,STREET,NaN,02108,G,Full Time,Not-at-Home,NaN,Boston College,2023-2024,"101 CHESTNUT ST, MA 02108",None,Unregistered
164866,101,LANARK,RD,NaN,02135,G,Full Time,Not-at-Home,NaN,Boston College,2023-2024,"101 LANARK RD, MA 02135",None,Unregistered
164870,103,CASS,STREET,NaN,02132,G,Full Time,Not-at-Home,NaN,Boston College,2023-2024,"103 CASS ST, MA 02132",None,Unregistered
164881,11,BRECK,AVE,#3,02135,G,Full Time,Not-at-Home,NaN,Boston College,2023-2024,"11 BRECK AVE, MA 02135","11 BRECK AVE, MA 02135",Registered (Fuzzy)


In [12]:
status_counts = student_df_unique['registration_status'].value_counts()
total = status_counts.sum()

summary_df = pd.DataFrame({
    'Count': status_counts,
    'Percentage': round((status_counts / total) * 100, 2)
})
print(summary_df)


                     Count  Percentage
registration_status                   
Unregistered         66557       86.29
Registered (Fuzzy)   10576       13.71


In [17]:
# Step 1: Build standardized address from SAM
def standardize_sam_address(row):
    parts = []
    if pd.notna(row["STREET_NUMBER"]):
        parts.append(str(row["STREET_NUMBER"]).strip())
    if pd.notna(row["STREET_BODY"]):
        parts.append(str(row["STREET_BODY"]).strip().upper())
    if pd.notna(row["STREET_SUFFIX_ABBR"]):
        parts.append(str(row["STREET_SUFFIX_ABBR"]).strip().upper())
    
    address = " ".join(parts)
    zip_code = str(row["ZIP_CODE"]).strip() if pd.notna(row["ZIP_CODE"]) else ""
    
    full_address = f"{address}, MA {zip_code}".upper().replace("  ", " ").strip()
    return full_address

# Load SAM dataset
sam_df = pd.read_csv("SAM.csv")

# Fix ZIP codes to ensure they are 5-digit strings with leading zeros
sam_df["ZIP_CODE"] = pd.to_numeric(sam_df["ZIP_CODE"], errors="coerce")
sam_df["ZIP_CODE"] = sam_df["ZIP_CODE"].fillna("").apply(lambda x: str(int(x)).zfill(5) if x != "" else "")

# Build standardized SAM address
sam_df["standardized_address"] = sam_df.apply(standardize_sam_address, axis=1)

# Step 2: Create dictionary for lookup
sam_address_to_id = sam_df.set_index("standardized_address")["SAM_ADDRESS_ID"].to_dict()

# Step 3: Map to student matched addresses (falling back to standardized_address if not matched)
def get_best_address(row):
    return row["matched_address"] if pd.notna(row["matched_address"]) else row["standardized_address"]

student_df_unique["SAM ID"] = student_df_unique.apply(lambda row: sam_address_to_id.get(get_best_address(row)), axis=1)

# Step 4: Save to CSV
student_df_unique.to_csv("student_addresses_with_sam_id.csv", index=False)


C:\Users\bens0\AppData\Local\Temp\ipykernel_18152\974554226.py:18: DtypeWarning: Columns (7,8,16,25,26) have mixed types. Specify dtype option on import or set low_memory=False.
  sam_df = pd.read_csv("SAM.csv")


In [19]:
# Total number of rows
total_rows = len(student_df_unique)

# Count how many rows got a SAM ID
matched_rows = student_df_unique["SAM ID"].notna().sum()

# Calculate percentage
matched_pct = (matched_rows / total_rows) * 100
unmatched_pct = 100 - matched_pct

# Print results
print(f"SAM ID matched: {matched_rows} out of {total_rows} rows ({matched_pct:.2f}%)")
print(f"No SAM ID match: {total_rows - matched_rows} rows ({unmatched_pct:.2f}%)")


SAM ID matched: 25951 out of 77133 rows (33.64%)
No SAM ID match: 51182 rows (66.36%)


In [29]:
# Count registered vs unregistered addresses by university
university_status = student_df_unique.groupby(['university', 'registration_status']).size().unstack(fill_value=0)
university_status['Total'] = university_status.sum(axis=1)
university_status['% Unregistered'] = round(
    (university_status['Unregistered'] / university_status['Total']) * 100, 2
)
university_status = university_status.sort_values('% Unregistered', ascending=False)
print(university_status)


registration_status                                 Registered (Fuzzy)  \
university                                                               
SHOWA Boston Institute                                               0   
St. John's Seminary                                                  0   
Northeastern Univerisity                                             0   
Boston Baptist College                                               0   
MA College of Art and Design                                         0   
MIT UAR Fall 2017-update                                             0   
MCSPHS University                                                    0   
MCPHHS University                                                    5   
Berklee College of Music                                             7   
Preliminary Tufts University-SMFA                                    8   
Boston College, WCAS                                                 1   
Simmons College                       

In [30]:
type_status = student_df_unique.groupby(['7. undergraduate (u) or graduate (g)', 'registration_status']).size().unstack(fill_value=0)
type_status['Total'] = type_status.sum(axis=1)
type_status['% Unregistered'] = round(
    (type_status['Unregistered'] / type_status['Total']) * 100, 2
)
print(type_status)


registration_status                   Registered (Fuzzy)  Unregistered  Total  \
7. undergraduate (u) or graduate (g)                                            
Exchange Student                                       0             1      1   
G                                                   3860         18768  22628   
G and U                                                0             1      1   
G and UG                                               0             2      2   
GR                                                    57           602    659   
Grad                                                 177           885   1062   
Graduate                                            1148         12949  14097   
U                                                   4423         25764  30187   
U and G                                                0             1      1   
U and Graduate                                         0             1      1   
UAG                         

In [31]:
zip_status = student_df_unique.groupby(['6e. zip', 'registration_status']).size().unstack(fill_value=0)
zip_status['Total'] = zip_status.sum(axis=1)
zip_status['% Unregistered'] = round(
    (zip_status['Unregistered'] / zip_status['Total']) * 100, 2
)
zip_status = zip_status.sort_values('% Unregistered', ascending=False)
print(zip_status)


registration_status  Registered (Fuzzy)  Unregistered  Total  % Unregistered
6e. zip                                                                     
0110                                  0             1      1          100.00
02301                                 0             1      1          100.00
02297                                 0             1      1          100.00
02293                                 0             1      1          100.00
02228                                 0             1      1          100.00
...                                 ...           ...    ...             ...
02122                               336           892   1228           72.64
02124                               580          1500   2080           72.12
02128                               636          1551   2187           70.92
02113                               218           467    685           68.18
02121                               390           770   1160           66.38

In [32]:
year_status = student_df_unique.groupby(['year', 'registration_status']).size().unstack(fill_value=0)
year_status['Total'] = year_status.sum(axis=1)
year_status['% Unregistered'] = round(
    (year_status['Unregistered'] / year_status['Total']) * 100, 2
)
print(year_status)


registration_status  Registered (Fuzzy)  Unregistered  Total  % Unregistered
year                                                                        
2016-2017                           499         11404  11903           95.81
2017-2018                           408         10295  10703           96.19
2018-2019                          3252          7220  10472           68.95
2019-2020                          1244         11391  12635           90.15
2020-2021                          1275          4628   5903           78.40
2021-2022                           683          3287   3970           82.80
2022-2023                          1733         10741  12474           86.11
2023-2024                          1485          9905  11390           86.96


In [33]:
# List of unregistered addresses
outreach_candidates = student_df_unique[student_df_unique['registration_status'] == 'Unregistered']

# Summary by university
outreach_summary = outreach_candidates.groupby('university').size().sort_values(ascending=False)
print(outreach_summary)

# Optional: Get top 100 unregistered addresses
top_unregistered_addresses = outreach_candidates[['standardized_address', 'university', 'year']].head(100)
print(top_unregistered_addresses)


university
Northeastern University                                  26661
University of Massachusetts-Boston                       11846
Boston University                                         5180
Boston College                                            4236
University of Massachusetts Boston                        1610
Suffolk University                                        1597
MCPHHS University                                         1503
Wentworth Institute of Technology                         1159
MCPHS University                                          1150
Berklee College of Music                                  1136
Simmons University                                         951
Urban College of Boston                                    945
Preliminary Tufts University-SMFA                          828
Tufts University                                           773
Simmons College                                            706
Emerson College                             